# Notebook 7: Prescriptive Constraints Validation

## 1. Introduction
In a real-world manufacturing environment, prescriptive recommendations must be **feasible**. We cannot suggest resource configurations that exceed the physical capacity of the factory or violate company policy (e.g., recommending 0 workers or 500 machines).

This notebook is dedicated to defining and validating the **Operational Guardrails** for the Prescriptive Engine.

### Key Objectives:
1. **Define Resource Bounds**: Establish the minimum and maximum allowable limits for workers, working hours, and machine counts.
2. **Implement Feasibility Logic**: Create a validation function that checks if a suggested configuration is operationally possible.
3. **Stress Test Constraints**: Verify that the validator correctly identifies both valid and invalid scenarios.

In [1]:
import pandas as pd
import numpy as np

print("Libraries imported.")

Libraries imported.


## 2. Defining Operational Bounds

Based on the Exploratory Data Analysis (EDA) from Notebook 2, we define the boundaries for our search space. We want the optimizer to search within a range that is realistic based on historical data but allows for some flexibility to meet targets.

In [2]:
# Operational Constraints Configuration
# These ranges are based on historical distributions seen in the dataset
RESOURCE_CONSTRAINTS = {
    'no_of_workers': {
        'min': 8,   # Based on min observed in dataset
        'max': 32,  # Based on max observed in dataset
        'step': 1
    },
    'working_hours': {
        'min': 250.0, # Based on lower end of typical weekly hours
        'max': 1500.0, # Based on upper end of observed hours
        'step': 8.0    # Hours usually change in shifts/blocks
    },
    'machine_count': {
        'min': 2,   # Minimum machines required for operation
        'max': 7,    # Maximum available machines in the unit
        'step': 1
    }
}

print("Resource constraints defined:")
for res, bounds in RESOURCE_CONSTRAINTS.items():
    print(f"{res}: Min={bounds['min']}, Max={bounds['max']}, Step={bounds['step']}")

Resource constraints defined:
no_of_workers: Min=8, Max=32, Step=1
working_hours: Min=250.0, Max=1500.0, Step=8.0
machine_count: Min=2, Max=7, Step=1


## 3. Feasibility Validator Implementation

The `is_feasible` function acts as a gatekeeper. The Optimizer will generate thousands of combinations; this function will instantly reject those that are impossible.

In [3]:
def is_feasible(config):
    """
    Checks if a resource configuration falls within the defined operational bounds.
    
    Args:
        config (dict): Dictionary containing 'no_of_workers', 'working_hours', 'machine_count'
    
    Returns:
        bool: True if the configuration is feasible, False otherwise.
    """
    for resource, bounds in RESOURCE_CONSTRAINTS.items():
        val = config.get(resource)
        if val is None:
            return False
        
        if val < bounds['min'] or val > bounds['max']:
            return False
            
    # Add specific business rules here if needed
    # Example: A rule that requires at least 1 machine for every 10 workers
    if config['no_of_workers'] > 10 and config['machine_count'] < 1:
        return False
        
    return True

print("Feasibility validator function defined.")

Feasibility validator function defined.


## 4. Constraint Stress Testing

We will now test the validator with a set of diverse scenarios to ensure it correctly identifies valid and invalid configurations.

In [4]:
# Test Scenarios
test_cases = [
    {
        'name': 'Typical Valid Config',
        'config': {'no_of_workers': 15, 'working_hours': 600, 'machine_count': 4},
        'expected': True
    },
    {
        'name': 'Too Few Workers',
        'config': {'no_of_workers': 2, 'working_hours': 600, 'machine_count': 4},
        'expected': False
    },
    {
        'name': 'Too Many Machines',
        'config': {'no_of_workers': 15, 'working_hours': 600, 'machine_count': 20},
        'expected': False
    },
    {
        'name': 'Excessive Working Hours',
        'config': {'no_of_workers': 15, 'working_hours': 5000, 'machine_count': 4},
        'expected': False
    },
    {
        'name': 'Boundary Min Case',
        'config': {'no_of_workers': 8, 'working_hours': 250, 'machine_count': 2},
        'expected': True
    },
    {
        'name': 'Boundary Max Case',
        'config': {'no_of_workers': 32, 'working_hours': 1500, 'machine_count': 7},
        'expected': True
    }
]

print(f"{'Scenario':<30} | {'Result':<10} | {'Status':<10}")
print("-" * 55)

for case in test_cases:
    result = is_feasible(case['config'])
    status = '✅ PASS' if result == case['expected'] else '❌ FAIL'
    print(f"{case['name']:<30} | {str(result):<10} | {status}")

Scenario                       | Result     | Status    
-------------------------------------------------------
Typical Valid Config           | True       | ✅ PASS
Too Few Workers                | False      | ✅ PASS
Too Many Machines              | False      | ✅ PASS
Excessive Working Hours        | False      | ✅ PASS
Boundary Min Case              | True       | ✅ PASS
Boundary Max Case              | True       | ✅ PASS


## 5. Summary
The resource boundaries are now defined and the `is_feasible` function is validated. 

These constraints will be used in **Notebook 8 (The Optimizer)** to prune the search space, ensuring that only realistic resource combinations are simulated and recommended.